## Topic: Pydantic Output Parser in LangChain

### 1. Introduction of PydanticOutputParser

- Definition:
    - PydanticOutputParser is a structured output parser in LangChain that uses Pydantic Model to enforce schema validation when processing LLM responses.


- Why use PydanticOutputParser?
    - 1. Strict Schema Enforcement:
        - Ensures that LLM responses follow a well-defined structured.

    - 2. Type Safety:
        - Automatically converts LLM outputs into Python objects.

    - 3. Easy Validation:
        - Uses Pydantic's built-in validation to catch incorrect or missing data.

    - 4. Seamless Integration:
        - Works well with other LangChain Components.


- use PydanticOutputParser import class:
    - from langchain_core.output_parsers import PydanticOutputParser

In [ ]:
"""         - The Two Superpowers of PydanticOutputParser

┌──────────────────────────────────────────────────────────────┐
│              PydanticOutputParser                             │
│                                                              │
│  SUPERPOWER 1: AUTO-GENERATED PROMPT INSTRUCTIONS            │
│  ─────────────────────────────────────────────               │
│  Your Pydantic model:                                        │
│    class User(BaseModel):                                    │
│        name: str = Field(description="Full name")            │
│        age: int = Field(description="Age in years")          │
│                                                              │
│  Auto-generates this for the LLM:                            │
│    "The output should be formatted as a JSON instance         │
│     that conforms to the JSON schema below.                   │
│     Properties:                                               │
│     - name (string): Full name                               │
│     - age (integer): Age in years"                           │
│                                                              │
│  You write the schema ONCE → LLM gets perfect instructions   │
│                                                              │
│  SUPERPOWER 2: AUTOMATIC VALIDATION                          │
│  ─────────────────────────────────────                       │
│  LLM returns: {"name": "John", "age": "thirty"}              │
│  Pydantic says:  ValidationError! "age" must be int!         │
│                                                              │
│  LLM returns: {"name": "John", "age": 30}                    │
│  Pydantic says:  Valid! Returns User(name="John", age=30)    │
│                                                              │
│  - Type safety guaranteed at runtime                         │
└──────────────────────────────────────────────────────────────┘
"""

In [ ]:
"""           - What Pydantic Validates Automatically: 

┌─────────────────────────────────────────────────────┐
│          PYDANTIC VALIDATION CHECKLIST              │
├──────────────────────┬──────────────────────────────┤
│ -  Type Checking     │ str, int, float, bool, list  │
│ -  Range Constraints │ ge=0, le=100, gt=0, lt=10    │
│ -  String Length     │ min_length=2, max_length=50  │
│ -  Pattern Matching  │ regex for emails, phones     │
│ -  Required Fields   │ Missing field → Error        │
│ -  Enum Validation   │ Only allowed values          │
│ -  Nested Objects    │ Validates sub-models too     │
│ -  List Types        │ list[int], list[str]         │
│ -  Custom Validators │ Your own validation logic    │
└──────────────────────┴──────────────────────────────┘

"""

In [ ]:
# Example 1:
# import require libraries
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from lanchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional


load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    task = "text-generation"
)

# initialize the Chat model object
model = ChatHuggingFace(llm = llm)

# Define the output schema using pydantic class
class Person(BaseModel):
    name: str = Field(
        description="Name of the Person",
        min_length=2,
        max_length=100
    )

    age: float = Field(
        description="Age of the Person",
        gt=0,          # Greater than 0
        le=99    # Less than or equal to
    )

    city: Optional(str) = Field(
        description = "Name of the City the person belong to",
        default = None
    )


# Now create a parser(PydanticOutputParser) that follow the pydantic class (Person)
parser = PydanticOutputParser(pydantic_object = Person)

# Create an Prompt template
template = PromptTemplate(
    template = "Generate the name, age and city of the factional {place} person \n {format_instruction}",

    input_variables = ["place"],

    partial_variables = {
        "format_instruction": parser.get_format_instructions()
    }

)

# prompt
prompt = template.invoke(
    {
        "place": "Bangladesh"
    }
)

response = model.invoke(prompt)

# parse the response
final_response = parser.parse(response.content)

print(final_response)

# the the Actual prompt the LLm received
print(prompt) 


In [ ]:
# Example 1: using chain component 
# import require libraries
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from lanchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional


load_dotenv()

# Define the model
llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    task = "text-generation"
)

# initialize the Chat model object
model = ChatHuggingFace(llm = llm)

# Define the output schema using pydantic class
class Person(BaseModel):
    name: str = Field(
        description="Name of the Person",
        min_length=2,
        max_length=100
    )

    age: float = Field(
        description="Age of the Person",
        gt=0,          # Greater than 0
        le=99    # Less than or equal to
    )

    city: Optional(str) = Field(
        description = "Name of the City the person belong to",
        default = None
    )


# Now create a parser(PydanticOutputParser) that follow the pydantic class (Person)
parser = PydanticOutputParser(pydantic_object = Person)

# Create an Prompt template
template = PromptTemplate(
    template = "Generate the name, age and city of the factional {place} person \n {format_instruction}",

    input_variables = ["place"],

    partial_variables = {
        "format_instruction": parser.get_format_instructions()
    }

)

# define a chain
chain = template | model | parser

# response
response = chain.invoke({
    "place": "Bangladesh"
})

print(response)

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────────┐
│           PydanticOutputParser INTERNAL FLOW                     │
│                                                                 │
│  1. YOU DEFINE THE MODEL                                        │
│     class Joke(BaseModel):                                      │
│         setup: str = Field(description="...")                   │
│         punchline: str = Field(description="...")               │
│                                                                 │
│  2. parser.get_format_instructions()                            │
│     ├── Reads your Pydantic model's schema                      │
│     ├── Extracts field names, types, descriptions               │
│     ├── Generates a JSON schema string                          │
│     └── Returns instructions to inject into the prompt          │
│                                                                 │
│  3. LLM RECEIVES THE PROMPT                                     │
│     "You are a comedian.                                        │
│      The output should be formatted as JSON...                   │
│      Schema: {setup: string, punchline: string, rating: int}    │
│      Tell me a joke about programming."                          │
│                                                                 │
│  4. LLM RETURNS RAW TEXT                                        │
│     '{"setup": "Why do programmers...", "punchline": "...",     │
│       "rating": 7}'                                              │
│                                                                 │
│  5. parser.parse(raw_text)                                      │
│     ├── Extracts JSON from the text (handles markdown, etc.)    │
│     ├── Calls Joke.model_validate_json(json_string)             │
│     ├── Pydantic checks:                                        │
│     │   ├── Is "setup" a string?                                │
│     │   ├── Is "punchline" a string?                            │
│     │   ├── Is "rating" an integer?                             │
│     │   ├── Is rating >= 1 and <= 10?                           │
│     │   └── Are all required fields present?                    │
│     └── Returns Joke(setup="...", punchline="...", rating=7)    │
│                                                                 │
│  6. YOUR CODE GETS A TYPED OBJECT                               │
│     result.setup      → "Why do programmers..."                 │
│     result.punchline  → "Because light attracts bugs!"          │
│     result.rating     → 7                                       │
└─────────────────────────────────────────────────────────────────┘

"""